In [0]:
%%capture --no-stderr
%pip install --quiet -U databricks-langchain langchain_core langgraph langgraph-prebuilt
dbutils.library.restartPython()

In [0]:
import os
os.environ['LANGSMITH_API_KEY'] = dbutils.secrets.get(scope="eo_scope", key="LANGSMITH_KEY")
os.environ['LANGSMITH_TRACING'] = "true"
os.environ['LANGSMITH_TRACING_V2'] = "true"
os.environ['LANGSMITH_PROJECT'] = "langchain-academy-module1"
os.environ['LANGSMITH_ENDPOINT'] = "https://api.smith.langchain.com"

In [0]:
from langgraph.graph import MessagesState
from langchain_core.messages import HumanMessage, SystemMessage

In [0]:
from databricks_langchain import ChatDatabricks
llm = ChatDatabricks(model='agents-demo-gpt4o')

In [0]:
def multiply(a: int, b: int) -> int:
  """
  multiply a and b

  Args:
    a: first number
    b: second number
  """
  return a * b

def add(a: int, b: int) -> int:
  """
  add a and b

  Args:
    a: first number
    b: second number
  """
  return a + b

def divide(a: int, b: int) -> float:
  """
  divide a and b

  Args:
    a: first number
    b: second number
  """
  return a / b

def subtract(a: int, b: int) -> int:
  """
  subtract a and b

  Args:
    a: first number
    b: second number
  """
  return a - b

In [0]:
llm_with_tools = llm.bind_tools([multiply, add, divide, subtract])

In [0]:
def assistant_node(state: MessagesState) -> MessagesState:

  system_prompt = SystemMessage(content='You are a helpful assistant tasked with doing arithmatic operations on set of inputs.')
  return {'messages' : llm_with_tools.invoke([system_prompt] + state['messages'])}

In [0]:
from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()

In [0]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
graph = StateGraph(MessagesState)

graph.add_node('assistant', assistant_node)
graph.add_node('tools', ToolNode([multiply, add, divide, subtract]))

graph.add_edge(START, 'assistant')
graph.add_conditional_edges('assistant', tools_condition)
graph.add_edge('tools', 'assistant')

app = graph.compile(checkpointer=memory)
app

In [0]:
config = {"configurable": {"thread_id": "1"}}

messages = [HumanMessage(content="Hello, what is 3 multiplied by 4, subtracted by 2 and devided by 5?")]
messages = app.invoke({"messages": messages}, config=config)
for m in messages['messages']:
    m.pretty_print()

In [0]:
messages = [HumanMessage(content="Now multiply that by 3.")]
messages = app.invoke({"messages": messages}, config=config)
for m in messages['messages']:
    m.pretty_print()